# Resume Analyzer - Exploratory Data Analysis (EDA)

This notebook explores the two primary raw datasets for the **resume-analyzer** system:
1. `data/raw/Resume.csv`: Resume text corpus labeled by professional domains/categories.
2. `data/raw/job_resume_fit.csv`: Paired resume and job description records with benchmark match scores and extracted skill lists.

Each section below contains an executive summary of findings followed by the execution code cell.

### 1. Environment & Library Setup

**Summary of Setup:**
- We import `pandas` for tabular data manipulation, `numpy` for numerical operations, and Python's built-in `ast` module to safely parse stringified lists into native Python `list` objects.
- The data directory is dynamically resolved so the notebook runs seamlessly whether executed from `notebooks/` or the project root.

In [ ]:
import ast
from pathlib import Path
import pandas as pd
import numpy as np

# Display formatting
pd.set_option('display.max_columns', 15)
pd.set_option('display.max_colwidth', 150)

# Resolve data directory
DATA_DIR = Path('data/raw') if Path('data/raw').exists() else Path('../data/raw')
print('Setup completed. Data directory:', DATA_DIR.resolve())

### 2. Resume.csv - Inspection: Shape, Columns, Dtypes, and Null Values

**Key Findings:**
- **Dataset Size**: Contains **2,484 resumes** across **4 columns** (`ID`, `Resume_str`, `Resume_html`, `Category`).
- **Data Types**: `ID` is an integer identifier (`int64`), while `Resume_str`, `Resume_html`, and `Category` are text strings (`object`).
- **Data Quality & Completeness**: There are **zero missing/null values** across all four columns (`0` nulls), indicating clean structural ingestion.

In [ ]:
resume_path = DATA_DIR / 'Resume.csv.gz'
df_resume = pd.read_csv(resume_path)

print(f"=== Resume.csv Shape ===\n{df_resume.shape[0]} rows, {df_resume.shape[1]} columns\n")
print("=== Columns ===")
print(df_resume.columns.tolist())
print("\n=== Data Types & Null Value Counts ===")
info_df = pd.DataFrame({
    'Dtype': df_resume.dtypes,
    'Null_Count': df_resume.isnull().sum(),
    'Null_Pct': (df_resume.isnull().sum() / len(df_resume)) * 100
})
print(info_df)

### 3. Resume.csv - Category Distribution

**Key Findings:**
- **Domain Diversity**: Spans **24 unique professional domains**.
- **Distribution Balance**: The top categories are well-represented with ~110–120 samples each (`INFORMATION-TECHNOLOGY` and `BUSINESS-DEVELOPMENT` lead with 120 each, followed by `ADVOCATE`, `CHEF`, `FINANCE`, `ENGINEERING`, and `ACCOUNTANT` with 118 each).
- **Under-represented Classes**: A few categories have fewer samples (`AGRICULTURE`: 63, `AUTOMOBILE`: 36, and `BPO`: 22), which should be accounted for during model evaluation or stratified cross-validation.

In [ ]:
print(f"=== Category Distribution (Total unique: {df_resume['Category'].nunique()}) ===")
category_counts = df_resume['Category'].value_counts()
category_pct = df_resume['Category'].value_counts(normalize=True) * 100

category_dist_df = pd.DataFrame({
    'Count': category_counts,
    'Percentage (%)': category_pct.round(2)
})
print(category_dist_df)

### 4. Resume.csv - Sample Rows & Text Formatting

**Key Findings:**
- `Resume_str` contains the full plain-text resume content with structural sections (Summary, Skills, Experience, Education) separated by newlines.
- `Resume_html` contains formatted DOM markup.
- Preprocessing will benefit from stripping non-standard punctuation, normalizing whitespace, and lowercasing.

In [ ]:
print("=== 3 Sample Rows from Resume.csv ===\n")
for idx, row in df_resume.head(3).iterrows():
    print(f"--- Sample #{idx + 1} | ID: {row['ID']} | Category: {row['Category']} ---")
    clean_str_snippet = ' '.join(str(row['Resume_str']).split())[:200]
    clean_html_snippet = ' '.join(str(row['Resume_html']).split())[:120]
    print(f"Resume_str Preview: {clean_str_snippet}...")
    print(f"Resume_html Preview: {clean_html_snippet}...\n")

### 5. job_resume_fit.csv - Inspection: Shape, Columns, Dtypes, and Null Values

**Key Findings:**
- **Dataset Size**: Contains **2,385 paired job-resume records** across **10 columns**.
- **Columns**: `ID`, `resume_text`, `job_text`, `category`, `job_required_skills`, `resume_skill_list`, `ai_matched_skills`, `ai_match_score`, `skill_string_match_score`, `fuzzy_match_score`.
- **Data Types**: All three score fields are floating-point values (`float64`), `ID` is an integer (`int64`), and text & skill fields are stored as `object` strings.
- **Data Quality**: **Zero missing/null values** across all columns.

In [ ]:
fit_path = DATA_DIR / 'job_resume_fit.csv.gz'
df_fit = pd.read_csv(fit_path)

print(f"=== job_resume_fit.csv Shape ===\n{df_fit.shape[0]} rows, {df_fit.shape[1]} columns\n")
print("=== Columns ===")
print(df_fit.columns.tolist())
print("\n=== Data Types & Null Value Counts ===")
fit_info_df = pd.DataFrame({
    'Dtype': df_fit.dtypes,
    'Null_Count': df_fit.isnull().sum(),
    'Null_Pct': (df_fit.isnull().sum() / len(df_fit)) * 100
})
print(fit_info_df)

### 6. job_resume_fit.csv - Summary Statistics for Match Scores

**Key Findings:**
- `ai_match_score` (AI-evaluated semantic match):
  - Mean: **44.04**, Std: **24.63**, Min: **0.00**, Max: **100.00**.
  - Demonstrates wide dynamic range covering poor fits (0%), moderate fits (median ~40.65%), and strong fits (100%).
- `skill_string_match_score` (Exact/tokenized skill overlap percentage):
  - Mean: **11.55**, Std: **7.49**, Min: **0.00**, Max: **41.67**.
  - Tends to be lower in absolute magnitude, reflecting the reality that exact keyword matches only capture a fraction of required skills.
- `fuzzy_match_score` (Levenshtein / fuzzy token similarity score):
  - Mean: **56.58**, Std: **6.78**, Min: **26.15**, Max: **84.41**.
  - Tightly grouped around 50–65%, providing a baseline string distance metric.

In [ ]:
score_cols = ['ai_match_score', 'skill_string_match_score', 'fuzzy_match_score']
print("=== Summary Statistics for Match Scores ===")
print(df_fit[score_cols].describe().round(2))

print("\n=== Extended Summary Metrics ===")
stats_extended = pd.DataFrame({
    'Mean': df_fit[score_cols].mean(),
    'Std': df_fit[score_cols].std(),
    'Median': df_fit[score_cols].median(),
    'IQR': df_fit[score_cols].quantile(0.75) - df_fit[score_cols].quantile(0.25),
    'Min': df_fit[score_cols].min(),
    'Max': df_fit[score_cols].max()
}).round(2)
print(stats_extended)

### 7. job_resume_fit.csv - Sample Rows & Real Text Format

**Key Findings:**
- Each row aligns a candidate's `resume_text` with an employer `job_text` under an assigned `category`.
- The snippets reveal that resumes contain rich experience bullet points while job descriptions detail role duties and qualification requirements.

In [ ]:
print("=== 3 Sample Rows from job_resume_fit.csv ===\n")
for idx, row in df_fit.head(3).iterrows():
    print(f"--- Record #{idx + 1} | ID: {row['ID']} | Category: {row['category']} ---")
    print(f"Scores -> AI Match: {row['ai_match_score']:.1f}% | Skill Overlap: {row['skill_string_match_score']:.1f}% | Fuzzy: {row['fuzzy_match_score']:.1f}%")
    resume_snippet = ' '.join(str(row['resume_text']).split())[:200]
    job_snippet = ' '.join(str(row['job_text']).split())[:200]
    print(f"Resume Text: {resume_snippet}...")
    print(f"Job Text:    {job_snippet}...\n")

### 8. Parsing Stringified Skill Lists with `ast.literal_eval`

**Key Findings:**
- In `job_resume_fit.csv`, columns `job_required_skills`, `resume_skill_list`, and `ai_matched_skills` are stored as literal string representations of Python lists (e.g. `"['python', 'sql']"`, dtype `object`).
- Attempting to index or check membership directly treats characters individually rather than skills.
- `ast.literal_eval` safely evaluates and parses these strings into native Python `list` objects without running arbitrary code.
- Below we demonstrate the before/after types, safely parse the columns, and inspect the resulting list distributions.

In [ ]:
skill_cols = ['job_required_skills', 'resume_skill_list', 'ai_matched_skills']

print("=== 1. Checking Raw String Representation Before Parsing ===")
sample_row = df_fit.iloc[0]
for col in skill_cols:
    raw_val = sample_row[col]
    print(f"{col}: type={type(raw_val).__name__} | value={raw_val[:75]}...")

print("\n=== 2. Parsing Safely with ast.literal_eval ===")
def safe_parse_skills(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return val if isinstance(val, list) else []

# Apply parsing across the skill columns
for col in skill_cols:
    parsed_col_name = f"{col}_parsed"
    df_fit[parsed_col_name] = df_fit[col].apply(safe_parse_skills)
    first_parsed = df_fit[parsed_col_name].iloc[0]
    print(f"\n{col} -> Parsed Type: {type(first_parsed).__name__} | Total Items: {len(first_parsed)}")
    print(f"Sample parsed skills: {first_parsed[:5]}")

print("\n=== 3. Parsed Skill List Length Statistics ===")
skill_lengths_df = pd.DataFrame({
    'Mean_Skills_Count': [df_fit[f"{col}_parsed"].apply(len).mean() for col in skill_cols],
    'Median_Skills_Count': [df_fit[f"{col}_parsed"].apply(len).median() for col in skill_cols],
    'Min_Skills_Count': [df_fit[f"{col}_parsed"].apply(len).min() for col in skill_cols],
    'Max_Skills_Count': [df_fit[f"{col}_parsed"].apply(len).max() for col in skill_cols]
}, index=skill_cols).round(2)
print(skill_lengths_df)

### 9. Summary & Actionable Recommendations for Resume Analyzer

1. **Domain Classification**:
   - 24 professional categories provide a good multi-class text classification baseline.
   - Smaller categories (`BPO`: 22, `AUTOMOBILE`: 36, `AGRICULTURE`: 63) should use stratified k-fold splits or class weighting.
2. **Skill Vocabulary & Lookup**:
   - The parsed skill lists (`job_required_skills`, `resume_skill_list`, `ai_matched_skills`) provide a rich domain-specific skills taxonomy for `app/data/`.
   - Direct skill overlap ratio: `len(set(resume_skills) & set(job_skills)) / len(set(job_skills))` is an intuitive, explainable metric to display in API responses.
3. **Scoring Model Benchmarking**:
   - `ai_match_score` is a continuous target (0–100%) that can be predicted via regression or learned ranking models in `app/ml/`.
   - A composite score combining semantic embedding similarity, exact skill match, and fuzzy string match will align well with real-world hiring evaluations.